# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- Sequnetial, Parallel 모두 가능
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.


- 즉, Data Pipeline 문법이다.

```
prompt | llm | parser
```

In [1]:
import sys
sys.path.append("..")

from common_config import llm_connect
model = "gpt-5.4"

temperatuyre = 0
max_tokens = 2086

llm = llm_connect(model = model)

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 1. 기본 체인: prompt | llm | parser

In [3]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# | 연산자로 체인 구성. prompt -> llm -> parser 순서대로 다음 칸으로 결과값 넘김. parser는 runnable parameter이므로 들어오는 값에 따라 변함. 셋 모두 사전에 정의해야 할 필요 있음.
chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 데이터베이스"}) # chain의 prompt의 topic 자리에 "벡터 데이터베이스" 가 들어가게 됨.
print(result)

벡터 데이터베이스는 텍스트·이미지·음성 같은 데이터를 숫자 벡터로 저장하고, 의미적으로 비슷한 항목을 빠르게 찾도록 최적화된 데이터베이스입니다.


숫자 벡터 -> 1차원 행렬.
텍스트·이미지·음성 같은 데이터를 숫자 벡터로 저장 : 벡터 임베딩.
벡터 데이터베이스는 1. 벡터 임베딩시킨 정보를 넣는 장소이고 
2. 넣기만 하는게 아니라 의미적으로 비슷한 항목을 빠르게 찾아준다(넣은 문장과 가장 관련성 높은 k개의 문서를 찾는다. top-k).

## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

In [4]:
for chunk in chain.stream({"topic": "임베딩"}):
    # flush=True -> 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩은 텍스트나 이미지 같은 데이터를 의미가 비슷할수록 서로 가까운 숫자 벡터로 바꿔 컴퓨터가 의미를 비교하고 활용할 수 있게 하는 표현 방식입니다.

In [5]:
for chunk in chain.stream({"topic": "임베딩"}):
    # flush=False -> Python이 출력 내용을 잠시 버퍼에 모아두었다가 나중에 보여줌. 버퍼에 머무르는 시간이 좀 더 길어 자연스러움.
    print(chunk, end="", flush=False)

임베딩은 텍스트나 이미지 같은 데이터를 컴퓨터가 의미적 유사성을 계산할 수 있도록 숫자 벡터로 변환한 표현입니다.

# 3. Runnable의 공통 실행 메서드(중요!!)
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- prompt → llm → parser → custom function 을 Runnable 객체로 파이프라인화 시키면 입력과 출력까지의 과정이 하나의 파이프라인이 됨.

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

In [11]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain = RunnablePassthrough()
result = chain.invoke({"topic": "RAG"})
print(result)

{'topic': 'RAG'}


In [ ]:
import base64
from IPython.display import Image, display

mermaid_code = """
flowchart LR
    A["입력값<br/>{'topic': 'RAG'}"] -->|"chain.invoke()"| B["RunnablePassthrough()<br/>(변환 없음)"]
    B -->|"입력 그대로 반환"| C["출력값<br/>{'topic': 'RAG'}"]
    C --> D["print(result)"]
"""

encoded = base64.urlsafe_b64encode(mermaid_code.encode("utf-8")).decode("ascii")
display(Image(url=f"https://mermaid.ink/img/{encoded}"))

- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [12]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
from langchain_core.runnables import RunnablePassthrough

upper=lambda x: x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper) # upper["word"].uppper()

result = chain.invoke({"word": "hello"})
print(result)  # {'word': 'hello', 'upper': 'HELLO'}

{'word': 'hello', 'upper': 'HELLO'}


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [8]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}이 뭔가요?")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix) # 함수를 객체로 만들어서 파이프라인 실행

result = chain.invoke({"topic": "LangGraph"})
print(result)

[결과] LangGraph는 **LLM(대규모 언어 모델) 기반 애플리케이션을 “그래프 구조”로 설계하고 실행할 수 있게 해주는 프레임워크**입니다.  
쉽게 말하면, 여러 단계의 AI 작업을 **노드(node)와 엣지(edge)** 로 나눠서, **상태를 유지하면서 복잡한 흐름**을 만들 수 있게 해줍니다.

## 왜 쓰나요?
일반적인 체인(chain) 방식은 순차적으로만 흐르기 쉬운데, 실제 AI 앱은 보통 더 복잡합니다.

예:
- 어떤 조건이면 다른 단계로 분기
- 중간 결과를 저장
- 사람이 승인하면 다음 단계 진행
- 에이전트가 도구를 여러 번 호출
- 실패하면 재시도하거나 다른 경로로 이동

LangGraph는 이런 흐름을 잘 다루도록 만들어졌습니다.

## 핵심 개념
- **State(상태)**: 현재까지의 대화, 도구 실행 결과, 변수 등을 저장
- **Node(노드)**: 하나의 작업 단위  
  예: LLM 호출, 검색, 계산, 검증
- **Edge(엣지)**: 다음에 어느 노드로 갈지 연결
- **Conditional routing**: 조건에 따라 다른 노드로 분기
- **Loop/iteration**: 필요하면 반복 수행
- **Persistence**: 실행 상태 저장 및 복구
- **Human-in-the-loop**: 사람 개입 가능

## LangChain과 차이
LangGraph는 보통 **LangChain 생태계의 일부/확장**으로 많이 언급됩니다.

- **LangChain**: LLM 앱을 만들기 위한 다양한 컴포넌트/체인 제공
- **LangGraph**: 그 위에서 **더 복잡하고 상태 기반인 워크플로우/에이전트 흐름**을 구성하는 데 강함

즉:
- 단순한 프롬프트 → 답변 정도면 LangChain만으로 충분할 수 있음
- 여러 단계, 조건 분기, 반복, 상태 관리가 필요하면 LangGraph가 유용함

## 어떤 데 쓰이나요?
- AI 에이전트
- 멀티스텝 질의응답 시스템
- RAG 파이프라인의 고도화
- 고객지원

## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

In [9]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

# 두 체인을 병렬로 실행하여 각각 한국어와 영어 설명 생성
parallel_chain = RunnableParallel(
    korean=chain_ko,
    english=chain_en
)

result = parallel_chain.invoke({"topic": "RAG"})
print("[한국어]", result["korean"])
print("[English]", result["english"])

[한국어] RAG는 **Retrieval-Augmented Generation**의 줄임말로, 한국어로는 보통 **검색 증강 생성**이라고 합니다.

쉽게 말하면,  
**LLM(대규모 언어 모델)이 답변을 만들기 전에 외부 자료를 먼저 찾아보고, 그 내용을 바탕으로 답변하는 방식**입니다.

## 왜 필요한가?
일반적인 언어 모델은 학습된 데이터만 바탕으로 답변하기 때문에:

- 최신 정보에 약할 수 있고
- 특정 회사 문서, 내부 지식, 매뉴얼 같은 **전용 자료**는 모를 수 있으며
- 사실과 다른 내용을 그럴듯하게 말하는 **환각(hallucination)** 문제가 생길 수 있습니다

RAG는 이런 문제를 줄이기 위해 사용됩니다.

## 어떻게 동작하나?
보통 다음 순서로 작동합니다:

1. **사용자가 질문함**
   - 예: “우리 회사 환불 정책이 뭐야?”

2. **관련 문서 검색**
   - 문서 DB, 위키, PDF, 매뉴얼 등에서 질문과 관련된 내용을 찾음

3. **찾은 문서를 LLM에 함께 전달**
   - 질문만 주는 것이 아니라, 검색된 참고 자료도 같이 넣어줌

4. **LLM이 참고 자료 기반으로 답변 생성**
   - 그래서 더 정확하고 근거 있는 답변을 만들 수 있음

## 예시
질문:
> “2024년 기준 우리 서비스 요금제 차이가 뭐야?”

RAG가 없다면:
- 모델이 기억에 의존해서 대답하거나
- 틀린 정보를 만들어낼 수 있음

RAG가 있다면:
- 최신 요금제 문서를 검색한 뒤
- 그 문서를 바탕으로 답변함

## 장점
- **최신 정보 반영 가능**
- **사내 문서/전문 자료 활용 가능**
- **환각 감소**
- **근거 기반 답변 제공 가능**

## 단점
- 검색이 잘못되면 답변도 부정확해질 수 있음
- 시스템 구성이 더 복잡함
- 문서 전처리, 벡터DB, 검색 품질 튜닝 등이 필요함

## 어디에 많이 쓰이나?
- 사내 챗봇
- 고객센터 자동응답
- 문서 검색 Q&A
-
[English] RAG stands

In [15]:
import base64
from IPython.display import Image, display

mermaid_code = """
flowchart LR
    IN["입력값<br/>{'topic': 'RAG'}"] --> PAR["RunnableParallel"]

    PAR --> P1["prompt_ko<br/>'{topic}을 한국어로 설명해줘'"]
    PAR --> P2["prompt_en<br/>'Explain {topic} in English'"]

    subgraph KO["chain_ko"]
        P1 --> L1["llm"] --> O1["StrOutputParser"]
    end

    subgraph EN["chain_en"]
        P2 --> L2["llm"] --> O2["StrOutputParser"]
    end

    O1 -->|"korean"| R["result<br/>{'korean': ..., 'english': ...}"]
    O2 -->|"english"| R

    R --> PR["print(result['korean'])<br/>print(result['english'])"]
"""

encoded = base64.urlsafe_b64encode(mermaid_code.encode("utf-8")).decode("ascii")
display(Image(url=f"https://mermaid.ink/img/{encoded}"))


## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요. 결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [10]:
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return {
        "text": text,
        "word_count": len(text.split())
    }

# 5. RunnableLambda로 함수 감싸기
word_count_runnable = RunnableLambda(count_words)

# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_count_runnable

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"])  # 설명의 단어 수
print("-"*20)
print(result["text"])        # LLM이 생성한 설명

NameError: name 'ChatOpenAI' is not defined